In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

port = pd.read_csv(OUT / "portfolio_B_price_only_500.csv")
rets = pd.read_csv(DATA_PROCESSED / "return_panel_500.csv", parse_dates=["date"])

port["ticker"] = port["ticker"].astype(str)
rets["ticker"] = rets["ticker"].astype(str)
port["report_date"] = pd.to_datetime(port["report_date"])
rets["date"] = pd.to_datetime(rets["date"])

pf = port.merge(rets, left_on=["report_date", "ticker"], right_on=["date", "ticker"], how="left")
pf["weighted_return"] = pf["weight"] * pf["return"]

bt = pf.groupby("report_date", as_index=False).agg(
    portfolio_return=("weighted_return", "sum"),
    n_names=("ticker", "nunique"),
    n_holdings=("selected", "sum")
)

bt["cum_return"] = (1 + bt["portfolio_return"].fillna(0)).cumprod() - 1
bt.to_csv(OUT / "backtest_B_price_only_500.csv", index=False)

print(bt.head())

  report_date  portfolio_return  n_names  n_holdings  cum_return
0  2015-02-28          0.030244      453          50    0.030244
1  2015-03-31          0.025854      453          50    0.056880
2  2015-04-30         -0.037793      453          50    0.016937
3  2015-05-31          0.056385      453          50    0.074277
4  2015-06-30          0.004209      454          50    0.078799
